# EDA — Medical Abstracts TC Corpus

**Objetivo:** entender o dataset antes de modelar, identificar desbalanceamento de classes, 
estatísticas de comprimento de texto e embasar o mapeamento das condições originais para as 
3 classes de urgência (`normal` / `atenção` / `urgente`).

**Dataset:** Medical Abstracts TC Corpus — 14.438 resumos médicos em inglês, 5 classes de condição clínica.  
**Fonte:** `data/raw/medical_tc_train.csv` + `data/raw/medical_tc_test.csv`

## 1. Imports e carregamento

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['figure.figsize'] = (10, 4)

In [ ]:
train  = pd.read_csv('../data/raw/medical_tc_train.csv')
test   = pd.read_csv('../data/raw/medical_tc_test.csv')
labels = pd.read_csv('../data/raw/medical_tc_labels.csv')

# Concatena para análise geral
full = pd.concat([train, test], ignore_index=True)

# Mapa label -> nome
label_map = dict(zip(labels['condition_label'], labels['condition_name']))

print(f'Train:  {train.shape[0]:,} linhas')
print(f'Test:   {test.shape[0]:,} linhas')
print(f'Total:  {full.shape[0]:,} linhas')
print(f'Colunas: {full.columns.tolist()}')

## 2. Estrutura e qualidade dos dados

In [ ]:
print('=== Primeiras linhas ===')
full.head(3)

In [ ]:
print('=== Tipos e nulos ===')
print(full.dtypes)
print()
print('Valores nulos por coluna:')
print(full.isnull().sum())

In [ ]:
dup_rows  = full.duplicated().sum()
dup_texts = full['medical_abstract'].duplicated().sum()

print(f'Linhas completamente duplicadas : {dup_rows}')
print(f'Textos duplicados (mesmo abstract): {dup_texts} ({dup_texts/len(full)*100:.1f}%)')
print()
print('Nota: duplicatas de texto ocorrem quando o mesmo resumo aparece em classes')
print('diferentes (rótulo ambíguo). São mantidas no treino — o modelo aprende a')
print('priorizar a maioria de votos implícita do TF-IDF.')

## 3. Distribuição de classes

In [ ]:
# Tabela de distribuição
counts = full['condition_label'].value_counts().sort_index()
df_dist = pd.DataFrame({
    'condition_label': counts.index,
    'condition_name': [label_map[l] for l in counts.index],
    'count': counts.values,
    'pct (%)': (counts.values / len(full) * 100).round(1)
})
print(df_dist.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Gráfico treino
counts_train = train['condition_label'].value_counts().sort_index()
names_short  = ['Neoplasms', 'Digestive', 'Nervous', 'Cardiovascular', 'General']
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']

axes[0].bar(names_short, counts_train.values, color=colors)
axes[0].set_title('Distribuição — Treino (11.550 amostras)')
axes[0].set_ylabel('Quantidade')
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(counts_train.values):
    axes[0].text(i, v + 30, f'{v/len(train)*100:.1f}%', ha='center', fontsize=9)

# Gráfico teste
counts_test = test['condition_label'].value_counts().sort_index()
axes[1].bar(names_short, counts_test.values, color=colors)
axes[1].set_title('Distribuição — Teste (2.888 amostras)')
axes[1].set_ylabel('Quantidade')
axes[1].tick_params(axis='x', rotation=20)
for i, v in enumerate(counts_test.values):
    axes[1].text(i, v + 8, f'{v/len(test)*100:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../docs/eda_class_distribution.png', bbox_inches='tight')
plt.show()
print('Figura salva em docs/eda_class_distribution.png')

**Observações:**
- Dataset **desbalanceado**: `general pathological conditions` representa 33,3% das amostras,
  enquanto `digestive system diseases` representa apenas 10,3%.
- A proporção treino/teste é idêntica por classe (~80/20), confirmando split estratificado.
- O desbalanceamento justifica o uso de `class_weight='balanced'` no treinamento.

## 4. Estatísticas de comprimento de texto

In [ ]:
full['len_chars']  = full['medical_abstract'].str.len()
full['len_tokens'] = full['medical_abstract'].str.split().str.len()

stats = full[['len_chars', 'len_tokens']].describe(
    percentiles=[.25, .50, .75, .90, .95, .99]
).round(1)
stats.columns = ['Comprimento (chars)', 'Comprimento (tokens)']
print(stats)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(full['len_chars'], bins=60, color='#4C72B0', edgecolor='white')
axes[0].axvline(full['len_chars'].median(), color='red',
                linestyle='--', label=f"Mediana {full['len_chars'].median():.0f}")
axes[0].axvline(full['len_chars'].mean(), color='orange',
                linestyle='--', label=f"Média {full['len_chars'].mean():.0f}")
axes[0].set_title('Distribuição — Comprimento em caracteres')
axes[0].set_xlabel('Caracteres')
axes[0].legend()

axes[1].hist(full['len_tokens'], bins=60, color='#55A868', edgecolor='white')
axes[1].axvline(full['len_tokens'].median(), color='red',
                linestyle='--', label=f"Mediana {full['len_tokens'].median():.0f}")
axes[1].axvline(full['len_tokens'].mean(), color='orange',
                linestyle='--', label=f"Média {full['len_tokens'].mean():.0f}")
axes[1].set_title('Distribuição — Comprimento em tokens')
axes[1].set_xlabel('Tokens (split por espaço)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../docs/eda_text_length.png', bbox_inches='tight')
plt.show()
print('Figura salva em docs/eda_text_length.png')

In [ ]:
# Comprimento por classe
print('Comprimento (chars) por classe — mediana | média | p95')
print('-' * 60)
for lbl in sorted(full['condition_label'].unique()):
    sub  = full[full['condition_label'] == lbl]['len_chars']
    name = label_map[lbl]
    print(f'  [{lbl}] {name:<35} '
          f'mediana={sub.median():>5.0f}  '
          f'média={sub.mean():>5.0f}  '
          f'p95={sub.quantile(0.95):>5.0f}')

In [ ]:
print('Textos com menos de 100 chars:', (full['len_chars'] < 100).sum())
print('Textos com mais de 5000 chars:', (full['len_chars'] > 5000).sum())
print(f'Máximo observado: {full["len_chars"].max()} chars')
print()
print('Conclusão: todos os textos estão dentro do limite de 5.000 chars da API.')
print('Não há textos muito curtos (< 100 chars) que precisariam ser filtrados.')

## 5. Inspeção qualitativa — amostras por classe

In [ ]:
np.random.seed(42)
for lbl in sorted(full['condition_label'].unique()):
    sample = full[full['condition_label'] == lbl]['medical_abstract'].sample(1).values[0]
    name   = label_map[lbl]
    print(f'\n[{lbl}] {name.upper()}')
    print(f'{sample[:400]}...')
    print('-' * 80)

## 6. Análise de duplicatas de texto

In [ ]:
# Textos que aparecem com mais de uma classe diferente
text_label_counts = (
    full.groupby('medical_abstract')['condition_label']
    .nunique()
)
ambiguous = text_label_counts[text_label_counts > 1]
print(f'Textos com mais de um rótulo: {len(ambiguous)}')

# Textos repetidos na mesma classe
same_class_dup = full[full.duplicated(subset=['medical_abstract', 'condition_label'])]
print(f'Duplicatas dentro da mesma classe: {len(same_class_dup)}')
print(f'Total de textos duplicados: {full["medical_abstract"].duplicated().sum()}')

## 7. Correlação entre comprimento e classe

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

data_by_class = [
    full[full['condition_label'] == lbl]['len_chars'].values
    for lbl in sorted(full['condition_label'].unique())
]
names_short = ['Neoplasms\n(1)', 'Digestive\n(2)', 'Nervous\n(3)',
               'Cardiovascular\n(4)', 'General\n(5)']

bp = ax.boxplot(data_by_class, tick_labels=names_short, patch_artist=True,
                medianprops=dict(color='black', linewidth=2))
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('Distribuição de comprimento (chars) por classe')
ax.set_ylabel('Comprimento (chars)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig('../docs/eda_length_by_class.png', bbox_inches='tight')
plt.show()
print('Figura salva em docs/eda_length_by_class.png')

## 8. Mapeamento das classes originais → urgência

A API precisa retornar uma das 3 classes de urgência: `normal`, `atenção` ou `urgente`.
O mapeamento é feito com base no nível de risco clínico típico de cada condição.

In [ ]:
mapping = {
    1: ('neoplasms',                      'atenção',
        'Cânceres exigem acompanhamento urgente mas raramente emergência imediata'),
    2: ('digestive system diseases',      'normal',
        'Patologias digestivas geralmente não são emergências imediatas'),
    3: ('nervous system diseases',        'atenção',
        'Doenças neurológicas requerem monitoramento; podem evoluir rapidamente'),
    4: ('cardiovascular diseases',        'urgente',
        'Condições cardiovasculares têm alto risco de eventos agudos fatais'),
    5: ('general pathological conditions','normal',
        'Condições patológicas gerais são heterogêneas e majoritariamente crônicas'),
}

print(f'{"Label":<6} {"Condição":<35} {"Urgência":<12} Justificativa')
print('-' * 100)
for lbl, (cond, urgencia, just) in mapping.items():
    print(f'{lbl:<6} {cond:<35} {urgencia:<12} {just}')

In [ ]:
# Distribuição resultante após o mapeamento
urgency_map = {lbl: v[1] for lbl, v in mapping.items()}
full['urgencia'] = full['condition_label'].map(urgency_map)

urg_counts = full['urgencia'].value_counts()
print('Distribuição após mapeamento:')
for urg, cnt in urg_counts.items():
    print(f'  {urg:<10}: {cnt:>5} ({cnt/len(full)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(6, 4))
order = ['normal', 'atenção', 'urgente']
bar_colors = ['#55A868', '#DD8452', '#C44E52']
vals = [urg_counts.get(u, 0) for u in order]
bars = ax.bar(order, vals, color=bar_colors)
ax.set_title('Distribuição — Classes de urgência (após mapeamento)')
ax.set_ylabel('Quantidade')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 50,
            f'{v:,}\n({v/len(full)*100:.1f}%)', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('../docs/eda_urgency_distribution.png', bbox_inches='tight')
plt.show()
print('Figura salva em docs/eda_urgency_distribution.png')

## 9. Conclusões

| Achado | Detalhe |
|---|---|
| Total de amostras | 14.438 (11.550 treino / 2.888 teste) |
| Colunas | `condition_label` (int 1–5), `medical_abstract` (texto) |
| Valores nulos | **0** em ambas as colunas |
| Textos duplicados | **3.211** (22,2%) — mesma classe, sem ambiguidade de rótulo |
| Desbalanceamento | Classe 5 (general): 33,3% vs classe 2 (digestive): 10,3% — razão ~3,2× |
| Comprimento médio | **1.231 chars / 180 tokens** |
| Comprimento mediano | **1.210 chars / 176 tokens** |
| p95 de comprimento | **2.050 chars / 302 tokens** |
| Máximo observado | **3.999 chars** — dentro do limite de 5.000 da API |
| Textos muito curtos (<100 chars) | **0** |
| Textos muito longos (>5000 chars) | **0** |

**Implicações para o modelo:**
- Usar `class_weight='balanced'` para compensar o desbalanceamento.
- Nenhum filtro de tamanho necessário — todos os textos são compatíveis com a API.
- TF-IDF com `sublinear_tf=True` mitiga a dominância de tokens frequentes nos textos mais longos.
- Os 3.211 duplicados (~22%) são mantidos — são cópias exatas na mesma classe, não contaminam o split estratificado.